# PDF to RDA DMP JSON

One prompt: the text of a DMP PDF plus the complete **maDMP 1.2** schema, with
strict instructions to follow the schema. Run first with `llama3.1:8b`, then the
same prompt with `gemma4:e4b` and `llama3.3:70b`.

| Step | What happens |
|---|---|
| 1 | Read sample 14 with pdfplumber |
| 2 | Load the schema — unchanged |
| 3 | Prompt → `llama3.1:8b` |
| 4 | Same prompt → `gemma4:e4b` |
| 5 | Same prompt → `llama3.3:70b` |
| 6 | Save all three |


In [1]:
import json
from pathlib import Path

if Path.cwd().name == "notebooks":
    import os
    os.chdir(Path.cwd().parent)

from dmpbridge.extractors import get_extractor
from dmpbridge.models.ollama import OllamaModel

PDF     = Path("data/input/pdfs/sample14.pdf")
SCHEMA  = Path("data/output/rda/maDMP-schema-1.2.json")
HOST    = "http://localhost:11434"
MODEL   = "llama3.1:8b"
MODEL_2 = "gemma4:e4b"
MODEL_3 = "llama3.3:70b"
OUT_DIR = Path("data/output/rda")


## Step 1 — Read the PDF


In [2]:
dmp_text = get_extractor("pdfplumber").extract(PDF)[0]["text"]

print(f"{len(dmp_text):,} characters\n")
print(dmp_text[:500])


12,930 characters

** Plan Overview **
_ A Data Management Plan created using DMPTool _
** DMP ID: ** https://doi.org/10.48321/D1CW23
** Title: ** Hakai Institute Juvenile Salmon Program Time Series
** Creator: ** Brett Johnson - ** ORCID: ** ++ 0000-0001-9317-0364 ++
** Affiliation: ** Hakai Institute
** Principal Investigator: ** Brett Johnson, Brian Hunt
** Data Manager: ** Brett Johnson, Tim van der Stap, Krystal Bachen
** Funder: ** Tula Foundation
** Template: ** Hakai Institute Data Management Plan
** Proje


## Step 2 — Load the schema

The schema is used exactly as published — nothing removed, nothing changed. Its
`$ref` pointers are resolved to the definitions they point to, so the model sees
the schema in one piece instead of copying `"$ref"` into its answer.


In [3]:
schema = json.loads(SCHEMA.read_text(encoding="utf-8"))
defs = schema["$defs"]


def resolve_refs(node):
    """Replace every $ref with the definition it points at. Content unchanged."""
    if isinstance(node, dict):
        if "$ref" in node:
            return resolve_refs(defs[node["$ref"].split("/")[-1]])
        return {k: resolve_refs(v) for k, v in node.items() if k != "$defs"}
    if isinstance(node, list):
        return [resolve_refs(v) for v in node]
    return node


schema_full = resolve_refs(schema)
schema_text = json.dumps(schema_full, separators=(",", ":"))

print(f"{len(schema_text):,} characters of schema, {len(defs)} definitions resolved")


37,332 characters of schema, 48 definitions resolved


## Step 3 — Prompt → llama3.1:8b

The schema is given to the model twice: as text in the prompt, and as Ollama's
`format`, which constrains the JSON it generates to that structure. Without the
constraint, `llama3.1:8b` fell into a loop on this prompt and never stopped;
`num_predict` is a cap in case it ever does again.


In [4]:
SYSTEM = """You convert Data Management Plans into RDA maDMP JSON.

Strict rules:
1. Use only the field names defined in the schema. Never add a key that is not in the schema.
2. Put every field exactly where the schema places it. The whole document is one top-level "dmp" object.
3. Where the schema lists allowed values, use one of them, spelled exactly as in the schema.
4. Take every value from the Data Management Plan text. Never copy example values from the schema.
5. Output only the JSON object. No explanation, no markdown."""

PROMPT = f"""Here is the RDA maDMP JSON schema (version 1.2). Follow it strictly:

{schema_text}

Here is the text of a Data Management Plan:

{dmp_text}

Generate one JSON object that strictly follows the schema above, filling in
whatever information the Data Management Plan text contains. Output only the JSON."""


def run(model):
    """Send the prompt to one model; the schema is also the output constraint."""
    llm = OllamaModel(model=model, host=HOST, num_ctx=32768, num_predict=8000)
    return json.loads(llm.complete(SYSTEM, PROMPT, schema=schema_full))


result_llama = run(MODEL)
print(json.dumps(result_llama, indent=2, ensure_ascii=False))


{
  "dmp": {
    "contact": {
      "contact_id": [
        {
          "identifier": "0000-0001-9317-0364",
          "type": "orcid"
        }
      ],
      "mbox": "brett.johnson@hakai.org",
      "name": "Brett Johnson"
    },
    "created": "2015-05-12T00:00:00Z",
    "dataset": [
      {
        "dataset_id": {
          "identifier": "https://doi.org/10.48321/D1CW23",
          "type": "handle"
        },
        "personal_data": "no",
        "sensitive_data": "unknown",
        "title": "Hakai Institute Juvenile Salmon Program Time Series",
        "type": "dataset"
      }
    ],
    "dmp_id": {
      "identifier": "https://doi.org/10.48321/D1CW23",
      "type": "handle"
    },
    "ethical_issues_exist": "no",
    "language": "eng",
    "modified": "2024-06-11T00:00:00Z",
    "title": "Hakai Institute Juvenile Salmon Program Time Series Data Management Plan"
  }
}


## Step 4 — Same prompt → gemma4:e4b


In [5]:
result_gemma = run(MODEL_2)
print(json.dumps(result_gemma, indent=2, ensure_ascii=False))


{
  "dmp": {
    "contact": {
      "contact_id": [
        {
          "identifier": "0000-0003-0644-4174",
          "type": "orcid"
        }
      ],
      "mbox": "N/A",
      "name": "Brett Johnson"
    },
    "created": "2015-05-12T00:00:00Z",
    "dataset": [
      {
        "dataset_id": {
          "identifier": "https://doi.org/10.48321/D1CW23",
          "type": "doi"
        },
        "personal_data": "unknown",
        "sensitive_data": "unknown",
        "title": "Hakai Institute Juvenile Salmon Program Time Series",
        "type": "raw data"
      },
      {
        "dataset_id": {
          "identifier": "N/A",
          "type": "handle"
        },
        "personal_data": "unknown",
        "sensitive_data": "unknown",
        "title": "Fatty acids from juvenile salmon",
        "type": "image"
      },
      {
        "dataset_id": {
          "identifier": "N/A",
          "type": "handle"
        },
        "personal_data": "unknown",
        "sensitive_data": "u

## Step 5 — Same prompt → llama3.3:70b

The 70B needs about 42 GB of VRAM, so the two smaller models are unloaded first.
This call takes a few minutes.


In [6]:
import subprocess
for m in (MODEL, MODEL_2):
    subprocess.run(["ollama", "stop", m], check=False)

result_llama33 = run(MODEL_3)
print(json.dumps(result_llama33, indent=2, ensure_ascii=False))


{
  "dmp": {
    "contact": {
      "contact_id": {
        "identifier": "0000-0001-9317-0364",
        "type": "orcid"
      },
      "mbox": "",
      "name": "Brett Johnson",
      "affiliation": [
        {
          "affiliation_id": {
            "identifier": "",
            "type": ""
          },
          "name": "Hakai Institute"
        }
      ]
    },
    "created": "2015-05-12T00:00:00Z",
    "dataset": [
      {
        "dataset_id": {
          "identifier": "https://doi.org/10.21966/1.566666",
          "type": "doi"
        },
        "personal_data": "no",
        "sensitive_data": "no",
        "title": "Hakai Institute Juvenile Salmon Program Time Series",
        "description": "The core of the Hakai JSP are the observations made at sea during seining operations, and the observations and measurements made in the lab during fish dissections.",
        "distribution": [
          {
            "data_access": "open",
            "title": "Hakai Institute Juvenile S

## Step 6 — Save all three


In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for model, result in ((MODEL, result_llama), (MODEL_2, result_gemma),
                      (MODEL_3, result_llama33)):
    out = OUT_DIR / f"{PDF.stem}.rda.{model.replace(':', '-')}.json"
    out.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{model:14} -> {out}")


llama3.1:8b    -> data\output\rda\sample14.rda.llama3.1-8b.json
gemma4:e4b     -> data\output\rda\sample14.rda.gemma4-e4b.json
llama3.3:70b   -> data\output\rda\sample14.rda.llama3.3-70b.json
